# Phase 1 — SVI Calibration & Baseline VRP Regression Falsification Harness

**Hypothesis under test:** Deribit BTC options exhibit a structural Volatility Risk Premium
$(IV^2 - RV^2) > 0$ that monetizes into statistically positive net returns after subtracting
Deribit option taker fees (0.03%), perpetual taker fees (0.05%), and bid/ask crossing slippage.

**Kill-gate criteria:** If $\beta \le 0$ or net-of-cost $\alpha \le 0$ ($p > 0.05$), the project HALTS.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from libs.core.svi_surface import SVIParameters, calibrate_svi_quasi_explicit, check_butterfly_arbitrage, svi_implied_vol, svi_total_variance
from tests.regression.test_phase1_vrp_regression import generate_btc_vrp_series


In [ ]:
# 1. SVI Slice Calibration Demonstration
tau = 14.0 / 365.0
target_iv = 0.65
w0 = (target_iv ** 2) * tau
true_params = SVIParameters(a=w0 - 0.1*w0*0.1, b=0.1*w0, rho=-0.20, m=0.0, sigma=0.10)
k_grid = np.linspace(-0.3, 0.3, 15)
w_quotes = svi_total_variance(k_grid, true_params)
calib = calibrate_svi_quasi_explicit(k_grid, w_quotes)
is_free, min_g = check_butterfly_arbitrage(calib)
print(f'Calibrated SVI: {calib}')
print(f'Arbitrage-free butterfly: {is_free} (min g = {min_g:.6f})')


In [ ]:
# 2. Rolling 14-day DTE Straddle Series Simulation
trades = generate_btc_vrp_series(n_days=120, seed=42)
df = pd.DataFrame([vars(t) for t in trades])
df[['cycle_id', 'entry_spot', 'iv_atm', 'rv_realized', 'iv2_minus_rv2', 'pnl_gross_usd', 'total_costs_usd', 'pnl_net_usd']].head(10)


In [ ]:
# 3. OLS Regression with Newey-West HAC Standard Errors
spread = df['iv2_minus_rv2'].values
mean_spread = float(np.mean(spread))
spread_centered = spread - mean_spread
X = sm.add_constant(spread_centered)
y = df['pnl_net_usd'].values
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 14})
print(model.summary())


In [ ]:
# 4. Kill-Gate 1 Decision
net_alpha = float(model.params[0])
beta = float(model.params[1])
alpha_p = float(model.pvalues[0])
beta_p = float(model.pvalues[1])
passed = (beta > 0 and beta_p < 0.05 and net_alpha > 0 and alpha_p < 0.05)
print(f'KILL GATE 1 VERDICT: {"PASSED -> PROCEED TO PHASE 2" if passed else "HALT PROJECT"}')
